# The Overfitting Problem

This notebook demonstrates **overfitting** - a critical pitfall in model calibration.

**Key concept**: A model with more parameters can fit the calibration data better, but may perform *worse* on new data. The goal is to capture the underlying signal, not the noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Import common setup
from calibration_common import (
    X_DOMAIN, L, H0, H1, K_TRUE, R_TRUE,
    X_OBS_STANDARD, H_OBS_STANDARD, H_TRUE_STANDARD,
    gw_model_1d, NOISE_STD
)

plt.rcParams['font.size'] = 11
plt.rcParams['figure.facecolor'] = 'white'

## The Setup: Same Model, Same Observations

We use the same 1D groundwater model and observations as the other demos:

$$h(x) = h_0 + (h_1 - h_0)\frac{x}{L} + \frac{R}{K} L \frac{x}{L}\left(1 - \frac{x}{L}\right)$$

The "true" parameters are K = 15 m/d, R = 0.001 m/d, but we pretend not to know them.

In [ ]:
# Our observations (from calibration_common)
x_obs = X_OBS_STANDARD
h_obs = H_OBS_STANDARD

# True model (we pretend not to know this)
x_true, h_true = gw_model_1d(K_TRUE, R_TRUE)

print("Observations:")
for i, (x, h) in enumerate(zip(x_obs, h_obs)):
    print(f"  Well {i+1}: x = {x:.0f} m, h = {h:.2f} m")

## The Temptation: More Parameters = Better Fit?

Instead of using our physics-based model (1-2 parameters), we could fit a polynomial:

| Model | Parameters | Description |
|-------|------------|-------------|
| Linear | 2 | $h = a + bx$ |
| Quadratic | 3 | $h = a + bx + cx^2$ |
| Cubic | 4 | $h = a + bx + cx^2 + dx^3$ |
| Quartic | 5 | $h = a + bx + cx^2 + dx^3 + ex^4$ |
| **Physics-based** | **1-2** | $h(x; K, R)$ from groundwater equation |

With 5 observations, we can fit up to a degree-4 polynomial *perfectly*. Is that good?

In [ ]:
def fit_polynomial(x_obs, h_obs, degree):
    """Fit a polynomial of given degree to observations."""
    coeffs = np.polyfit(x_obs, h_obs, degree)
    return coeffs

def evaluate_polynomial(coeffs, x):
    """Evaluate polynomial at given x values."""
    return np.polyval(coeffs, x)

def calc_rmse(h_obs, h_pred):
    """Calculate RMSE."""
    return np.sqrt(np.mean((h_obs - h_pred)**2))

# Fit polynomials of increasing degree
degrees = [1, 2, 3, 4]  # Linear to quartic
poly_fits = {}

print("Polynomial fits to calibration data:")
print("-" * 50)
for deg in degrees:
    coeffs = fit_polynomial(x_obs, h_obs, deg)
    h_pred_cal = evaluate_polynomial(coeffs, x_obs)
    rmse_cal = calc_rmse(h_obs, h_pred_cal)
    poly_fits[deg] = coeffs
    print(f"  Degree {deg}: RMSE = {rmse_cal:.4f} m ({deg+1} parameters)")

# Also fit physics-based model (optimize K)
from scipy.optimize import minimize_scalar

def physics_rmse(K):
    _, h_pred = gw_model_1d(K, R_TRUE, x=x_obs)
    return calc_rmse(h_obs, h_pred)

result = minimize_scalar(physics_rmse, bounds=(1, 100), method='bounded')
K_calibrated = result.x
rmse_physics = result.fun
print(f"\n  Physics model: RMSE = {rmse_physics:.4f} m (1 parameter, K={K_calibrated:.1f} m/d)")

## The Problem: Validation on New Data

A model should generalize to new data, not just fit the calibration data. Let's generate **validation observations** at different locations.

In [ ]:
# Validation observations (different locations, same true model)
np.random.seed(123)  # Different seed for validation
x_val = np.array([100, 250, 450, 600, 800, 950])
_, h_true_val = gw_model_1d(K_TRUE, R_TRUE, x=x_val)
h_val = h_true_val + np.random.normal(0, NOISE_STD, len(x_val))

print("Validation observations:")
for i, (x, h) in enumerate(zip(x_val, h_val)):
    print(f"  Well V{i+1}: x = {x:.0f} m, h = {h:.2f} m")

In [ ]:
# Evaluate all models on validation data
print("\nModel performance comparison:")
print("=" * 60)
print(f"{'Model':<20} {'Cal RMSE [m]':>15} {'Val RMSE [m]':>15}")
print("-" * 60)

results = []

for deg in degrees:
    coeffs = poly_fits[deg]
    # Calibration performance
    h_pred_cal = evaluate_polynomial(coeffs, x_obs)
    rmse_cal = calc_rmse(h_obs, h_pred_cal)
    # Validation performance
    h_pred_val = evaluate_polynomial(coeffs, x_val)
    rmse_val = calc_rmse(h_val, h_pred_val)
    
    results.append({'model': f'Polynomial deg {deg}', 'cal': rmse_cal, 'val': rmse_val, 'deg': deg})
    print(f"{'Polynomial deg ' + str(deg):<20} {rmse_cal:>15.4f} {rmse_val:>15.4f}")

# Physics model
_, h_pred_cal_phys = gw_model_1d(K_calibrated, R_TRUE, x=x_obs)
_, h_pred_val_phys = gw_model_1d(K_calibrated, R_TRUE, x=x_val)
rmse_cal_phys = calc_rmse(h_obs, h_pred_cal_phys)
rmse_val_phys = calc_rmse(h_val, h_pred_val_phys)
results.append({'model': 'Physics (K only)', 'cal': rmse_cal_phys, 'val': rmse_val_phys, 'deg': 'phys'})
print(f"{'Physics (K only)':<20} {rmse_cal_phys:>15.4f} {rmse_val_phys:>15.4f}")

print("-" * 60)
print("\nObservation: Higher-degree polynomials fit calibration data")
print("better but may perform WORSE on validation data!")

## Visualizing Overfitting

In [ ]:
def plot_overfitting_comparison(figsize=(10, 5)):
    """Plot showing overfitting vs physics-based model."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    x_plot = np.linspace(0, L, 200)
    
    # Left: All polynomial fits
    ax = axes[0]
    ax.plot(x_true, h_true, 'k-', linewidth=2, label='True model', alpha=0.5)
    
    colors = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']
    for deg, color in zip(degrees, colors):
        h_plot = evaluate_polynomial(poly_fits[deg], x_plot)
        ax.plot(x_plot, h_plot, color=color, linewidth=1.5, 
               label=f'Degree {deg} ({deg+1} params)', linestyle='--')
    
    ax.scatter(x_obs, h_obs, c='black', s=80, zorder=5, marker='o',
              edgecolor='white', linewidth=1.5, label='Calibration obs')
    ax.scatter(x_val, h_val, c='red', s=80, zorder=5, marker='s',
              edgecolor='white', linewidth=1.5, label='Validation obs')
    
    ax.set_xlabel('Distance x [m]')
    ax.set_ylabel('Hydraulic head h [m]')
    ax.set_title('Polynomial Fits: More Parameters ≠ Better', fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, L)
    
    # Right: Physics model vs overfit polynomial
    ax = axes[1]
    ax.plot(x_true, h_true, 'k-', linewidth=2, label='True model', alpha=0.5)
    
    # Physics-based
    _, h_phys = gw_model_1d(K_calibrated, R_TRUE)
    ax.plot(X_DOMAIN, h_phys, 'b-', linewidth=2, label=f'Physics (K={K_calibrated:.1f})')
    
    # Overfit polynomial (degree 4)
    h_poly4 = evaluate_polynomial(poly_fits[4], x_plot)
    ax.plot(x_plot, h_poly4, 'r--', linewidth=2, label='Degree 4 polynomial')
    
    ax.scatter(x_obs, h_obs, c='black', s=80, zorder=5, marker='o',
              edgecolor='white', linewidth=1.5, label='Calibration')
    ax.scatter(x_val, h_val, c='green', s=80, zorder=5, marker='s',
              edgecolor='white', linewidth=1.5, label='Validation')
    
    ax.set_xlabel('Distance x [m]')
    ax.set_ylabel('Hydraulic head h [m]')
    ax.set_title('Physics Model vs Overfitting', fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, L)
    
    plt.tight_layout()
    return fig, axes

fig, axes = plot_overfitting_comparison()
plt.show()

## The Bias-Variance Trade-off

This is a fundamental concept in model fitting:

| | Simple Model (Underfitting) | Complex Model (Overfitting) |
|---|---|---|
| **Bias** | High (misses true pattern) | Low (can fit anything) |
| **Variance** | Low (stable predictions) | High (sensitive to noise) |
| **Calibration fit** | Poor | Excellent |
| **Validation fit** | Poor | Often poor |
| **Generalization** | Poor | Poor |

The **optimal model** balances bias and variance.

In [ ]:
def plot_bias_variance(figsize=(8, 5)):
    """Plot showing bias-variance trade-off."""
    fig, ax = plt.subplots(figsize=figsize)
    
    # Extract results
    models = [r['model'] for r in results]
    cal_rmse = [r['cal'] for r in results]
    val_rmse = [r['val'] for r in results]
    
    x_pos = np.arange(len(models))
    width = 0.35
    
    bars1 = ax.bar(x_pos - width/2, cal_rmse, width, label='Calibration RMSE', 
                   color='#3498DB', alpha=0.8)
    bars2 = ax.bar(x_pos + width/2, val_rmse, width, label='Validation RMSE',
                   color='#E74C3C', alpha=0.8)
    
    ax.set_xlabel('Model Complexity →')
    ax.set_ylabel('RMSE [m]')
    ax.set_title('The Overfitting Problem: Calibration vs Validation', fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Linear', 'Quadratic', 'Cubic', 'Quartic', 'Physics'], rotation=15)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add annotation
    ax.annotate('Best calibration fit\n(but worst validation!)', 
               xy=(3, cal_rmse[3]), xytext=(2, 0.25),
               arrowprops=dict(arrowstyle='->', color='gray'),
               fontsize=9, ha='center')
    
    ax.annotate('Best overall\n(physics-based)', 
               xy=(4, val_rmse[4]), xytext=(4, 0.35),
               arrowprops=dict(arrowstyle='->', color='green'),
               fontsize=9, ha='center', color='green')
    
    plt.tight_layout()
    return fig, ax

fig, ax = plot_bias_variance()
plt.show()

## Animation: Watching Overfitting Happen

In [ ]:
def create_overfitting_animation(figsize=(10, 5)):
    """Create animation showing progressive overfitting."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    x_plot = np.linspace(0, L, 200)
    
    # Get y limits
    y_min = min(h_obs.min(), h_val.min()) - 1
    y_max = max(h_obs.max(), h_val.max()) + 1
    
    all_models = [(1, 'Linear (2 params)'), 
                  (2, 'Quadratic (3 params)'),
                  (3, 'Cubic (4 params)'),
                  (4, 'Quartic (5 params)'),
                  ('phys', 'Physics (1 param)')]
    
    def animate(frame):
        for ax in axes:
            ax.clear()
        
        idx = min(frame, len(all_models) - 1)
        deg, label = all_models[idx]
        
        # Left plot: Model fit
        ax = axes[0]
        ax.plot(x_true, h_true, 'k-', linewidth=2, alpha=0.3, label='True')
        
        if deg == 'phys':
            _, h_fit = gw_model_1d(K_calibrated, R_TRUE)
            ax.plot(X_DOMAIN, h_fit, 'g-', linewidth=2.5, label=label)
            color = 'green'
        else:
            h_fit = evaluate_polynomial(poly_fits[deg], x_plot)
            colors = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']
            color = colors[deg-1]
            ax.plot(x_plot, h_fit, color=color, linewidth=2.5, label=label)
        
        ax.scatter(x_obs, h_obs, c='black', s=100, zorder=5, marker='o',
                  edgecolor='white', linewidth=2, label='Calibration')
        ax.scatter(x_val, h_val, c='red', s=80, zorder=4, marker='s',
                  edgecolor='white', linewidth=1.5, alpha=0.7, label='Validation')
        
        ax.set_xlim(0, L)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel('Distance x [m]')
        ax.set_ylabel('Hydraulic head h [m]')
        ax.set_title(f'Current Model: {label}', fontweight='bold', fontsize=11)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Right plot: RMSE comparison
        ax = axes[1]
        
        models_shown = all_models[:idx+1]
        x_pos = np.arange(len(models_shown))
        width = 0.35
        
        cal_vals = []
        val_vals = []
        labels_shown = []
        
        for d, lbl in models_shown:
            if d == 'phys':
                _, hp_c = gw_model_1d(K_calibrated, R_TRUE, x=x_obs)
                _, hp_v = gw_model_1d(K_calibrated, R_TRUE, x=x_val)
            else:
                hp_c = evaluate_polynomial(poly_fits[d], x_obs)
                hp_v = evaluate_polynomial(poly_fits[d], x_val)
            cal_vals.append(calc_rmse(h_obs, hp_c))
            val_vals.append(calc_rmse(h_val, hp_v))
            labels_shown.append(lbl.split()[0])
        
        ax.bar(x_pos - width/2, cal_vals, width, label='Calibration', color='#3498DB', alpha=0.8)
        ax.bar(x_pos + width/2, val_vals, width, label='Validation', color='#E74C3C', alpha=0.8)
        
        ax.set_ylabel('RMSE [m]')
        ax.set_title('Model Performance', fontweight='bold', fontsize=11)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels_shown, fontsize=9)
        ax.legend(loc='upper left', fontsize=9)
        ax.set_ylim(0, 0.6)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Highlight current model
        if len(x_pos) > 0:
            ax.axvline(x_pos[-1], color=color, linewidth=2, alpha=0.3)
        
        plt.tight_layout()
        return []
    
    # Frames: show each model + hold at end
    n_frames = len(all_models) + 3
    anim = animation.FuncAnimation(fig, animate, frames=n_frames, interval=1200, blit=True)
    plt.close(fig)
    return anim, fig

anim, fig = create_overfitting_animation()
HTML(anim.to_jshtml())

## Save Animation

In [ ]:
# Recreate and save
anim, fig = create_overfitting_animation(figsize=(10, 4.5))

gif_path = 'overfitting_animation.gif'
anim.save(gif_path, writer='pillow', fps=0.8, dpi=100)
print(f"Animation saved to {gif_path}")

import os
size_kb = os.path.getsize(gif_path) / 1024
print(f"File size: {size_kb:.0f} KB")

## Key Takeaways

1. **More parameters ≠ better model** - Complex models can fit calibration data perfectly but fail on new data

2. **Overfitting fits noise, not signal** - The model learns the random measurement errors rather than the underlying physics

3. **Physics-based models are more robust** - They encode physical constraints that prevent unrealistic behavior

4. **Always validate on independent data** - Split your data: calibrate on some, validate on the rest

5. **Parsimony principle** - Prefer simpler models unless complexity is justified by improved validation performance